# Load Step

This notebook writes the transformed sales data to a Delta table in Unity Catalog.

**Target table**: `${catalog}.${schema}.sales_delta` (parameterized via widgets)

**Write mode**: `OVERWRITE` (idempotent — safe to re-run)

In [0]:
# Databricks notebook source
# --- Load: Write transformed data to Delta table ---

# Read parameters (with defaults)
_widgets = dbutils.widgets.getAll()
catalog = _widgets.get("catalog", "workspace")
schema = _widgets.get("schema", "default")
table_name = _widgets.get("table_name", "sales_delta")

full_table_name = f"{catalog}.{schema}.{table_name}"

print(f"Writing to table: {full_table_name}")

# Ensure the schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Get the transformed data from the temp view
transformed_sales_df = spark.table("transformed_sales")

# Write to Delta table with overwrite mode (idempotent)
(
    transformed_sales_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

print(f"\nSuccessfully wrote {transformed_sales_df.count()} rows to {full_table_name}")

# Verify the write by reading back
result = spark.table(full_table_name)
print(f"\nVerification — row count in table: {result.count()}")
result.printSchema()
result.show(5, truncate=False)

print("Load step completed successfully.")